# 02 - Fine-tune ArTST v3 (Audio → Diacritized Text)

**Base Model:** `MBZUAI/artst_asr_v3` (already fine-tuned SpeechT5)
**Architecture:** SpeechT5 - Audio to Text
**Current Score:** 0.43 WER

**Techniques:**
- Further fine-tuning on KSSA audio dataset
- Mixed Precision (FP16)
- Gradient Checkpointing

In [1]:
!pip install -q transformers torch accelerate tqdm datasets librosa soundfile

In [2]:
# Setup
import os, sys, json, re, torch, gc, subprocess, zipfile
from datetime import datetime
from tqdm import tqdm
from pathlib import Path
import librosa
import warnings
warnings.filterwarnings('ignore')

IS_RUNPOD = False  # Cloud detection removed
PROJECT_DIR = Path('.')
sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / 'Public_Data_TrainDev'
TRAIN_DIR = DATA_DIR / 'Train'
TRAIN_AUDIO_DIR = TRAIN_DIR / 'train_audio'
TRAIN_TEXT_DIR = TRAIN_DIR / 'train_text'
DEV_INPUT = DATA_DIR / 'dev input-output' / 'Dev_input.json'
DEV_OUTPUT = DATA_DIR / 'dev input-output' / 'Dev_output.json'
DEV_AUDIO_DIR = DATA_DIR / 'Dev' / 'dev_audio'
TEST_DIR = PROJECT_DIR / 'Test'
TEST_INPUT = TEST_DIR / 'test_input.json'
TEST_AUDIO_DIR = TEST_DIR / 'test_audio'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
SUBMISSION_DIR = PROJECT_DIR / 'submissions'
CHECKPOINTS_DIR = PROJECT_DIR / 'checkpoints' / 'artst_ft'
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
SUBMISSION_DIR.mkdir(exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Environment: 'Local' | Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB)")

def clear_gpu():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()
        print(f"GPU Memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

Environment: Local | Device: cuda
GPU: NVIDIA RTX A5000 (23.5 GB)


In [3]:
# Load Training Data (audio + text pairs)
import pandas as pd

train_tsv = TRAIN_DIR / 'train_list.tsv'
train_df = pd.read_csv(train_tsv, sep='\t')
print(f"Training samples: {len(train_df)}")

def load_training_data():
    data = []
    for idx, row in tqdm(train_df.iterrows(), total=len(train_df), desc="Loading data"):
        stem = row['stem']
        audio_path = TRAIN_AUDIO_DIR / f"{stem}.mp3"
        text_path = TRAIN_TEXT_DIR / f"{stem}.txt"
        
        if audio_path.exists() and text_path.exists():
            with open(text_path, 'r', encoding='utf-8') as f:
                diacritized = f.read().strip()
            data.append({
                'id': stem,
                'audio_path': str(audio_path),
                'text_diacritized': diacritized
            })
    return data

train_data = load_training_data()
print(f"Loaded {len(train_data)} audio-text pairs")

Training samples: 2327


Loading data: 100%|██████████| 2327/2327 [00:16<00:00, 137.77it/s]

Loaded 2327 audio-text pairs


In [12]:
# Load Model
from transformers import SpeechT5Processor, SpeechT5ForSpeechToText, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset, Audio

MODEL_NAME = 'MBZUAI/artst_asr_v3'
MODEL_KEY = 'artst_ft'

print(f"Loading {MODEL_NAME}...")
processor = SpeechT5Processor.from_pretrained(MODEL_NAME)
model = SpeechT5ForSpeechToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32  # Float32 for training stability
).to(device)

model.gradient_checkpointing_enable()
print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

Loading MBZUAI/artst_asr_v3...


Loading weights:   0%|          | 0/369 [00:00<?, ?it/s]

SpeechT5ForSpeechToText LOAD REPORT from: MBZUAI/artst_asr_v3
Key                                                  | Status     |  | 
-----------------------------------------------------+------------+--+-
speecht5.encoder.prenet.pos_sinusoidal_embed.weights | UNEXPECTED |  | 
speecht5.decoder.prenet.embed_positions.weights      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: 151,258,752 parameters


In [13]:
# Prepare Dataset
MAX_AUDIO_LENGTH = 16000 * 30  # 30 seconds max
MAX_TEXT_LENGTH = 256

def load_audio(path, sr=16000):
    try:
        audio, _ = librosa.load(path, sr=sr)
        return audio
    except:
        return None

def preprocess_function(examples):
    audios = [load_audio(p) for p in examples['audio_path']]
    
    # Process audio
    inputs = processor(
        audio=[a[:MAX_AUDIO_LENGTH] if a is not None else [] for a in audios],
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    
    # Process text labels
    labels = processor.tokenizer(
        examples['text_diacritized'],
        max_length=MAX_TEXT_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    return {
        'input_values': inputs['input_values'],
        'labels': labels['input_ids']
    }

# Create dataset (use subset for memory)
train_subset = train_data[:500]  # Start with subset, increase if memory allows
train_dataset = Dataset.from_list(train_subset)
train_dataset = train_dataset.map(
    preprocess_function, 
    batched=True, 
    batch_size=8,
    remove_columns=['id', 'audio_path', 'text_diacritized']
)

print(f"Train: {len(train_dataset)} samples")

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Train: 500 samples


In [14]:
# Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINTS_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=5e-6,  # Very low LR for fine-tuning
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=False,  # Keep float32 for stability
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    logging_steps=20,
    report_to="none",
    predict_with_generate=True,
    generation_max_length=MAX_TEXT_LENGTH,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [15]:
# Custom Data Collator
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2Seq:
    processor: Any
    
    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [torch.tensor(f['input_values']) for f in features]
        labels = [torch.tensor(f['labels']) for f in features]
        
        # Pad inputs
        input_features = torch.nn.utils.rnn.pad_sequence(input_features, batch_first=True)
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=-100)
        
        return {
            'input_values': input_features,
            'labels': labels
        }

data_collator = DataCollatorSpeechSeq2Seq(processor)

In [16]:
# Train
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

clear_gpu()
checkpoint = None
checkpoints = list(CHECKPOINTS_DIR.glob('checkpoint-*'))
if checkpoints:
    checkpoint = str(max(checkpoints, key=lambda x: int(x.name.split('-')[1])))
    print(f"Resuming from: {checkpoint}")

trainer.train(resume_from_checkpoint=checkpoint)

GPU Memory: 0.58 GB


Step,Training Loss
20,113.406055
40,45.823178
60,39.307208
80,35.116684


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=96, training_loss=54.305704752604164, metrics={'train_runtime': 396.8559, 'train_samples_per_second': 3.78, 'train_steps_per_second': 0.242, 'total_flos': 2.5370474193907354e+17, 'train_loss': 54.305704752604164, 'epoch': 3.0})

In [17]:
# Save
final_path = CHECKPOINTS_DIR / 'final'
trainer.save_model(str(final_path))
processor.save_pretrained(str(final_path))
print(f"Saved to: {final_path}")
clear_gpu()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to: ./checkpoints/artst_ft/final
GPU Memory: 1.73 GB


In [4]:
clear_gpu()

GPU Memory: 0.00 GB


## Inference on Dev and Test Sets

In [18]:
@torch.inference_mode()
def transcribe_diacritized(audio_path, fallback_text):
    audio = load_audio(audio_path)
    if audio is None:
        return fallback_text
    
    inputs = processor(audio=audio, sampling_rate=16000, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    generated_ids = model.generate(**inputs, max_length=MAX_TEXT_LENGTH)
    return processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

In [19]:
# Dev Set
with open(DEV_INPUT, 'r', encoding='utf-8') as f: dev_input = json.load(f)
CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_checkpoint.json"

def load_checkpoint():
    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_checkpoint(predictions):
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

processed_ids, dev_predictions = load_checkpoint()
print(f"Dev: {len(dev_input)} samples, {len(processed_ids)} already done")

for item in tqdm(dev_input, desc="Dev Set"):
    if item['id'] in processed_ids: continue
    try:
        audio_path = DEV_AUDIO_DIR / f"{item['id']}.mp3"
        result = transcribe_diacritized(audio_path, item['text_undiacritized'])
        dev_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_checkpoint(dev_predictions)
    except Exception as e:
        dev_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_checkpoint(dev_predictions)

with open(OUTPUT_DIR / f"{MODEL_KEY}_dev_predictions.json", 'w', encoding='utf-8') as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=2)

Dev: 260 samples, 0 already done


Dev Set:   0%|          | 0/260 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
Caching is incompatible with gradient checkpointing in SpeechT5DecoderLayer. Setting `past_key_values=None`.
Dev Set: 100%|██████████| 260/260 [16:42<00:00,  3.86s/it]


In [ ]:
# Test Set
with open(TEST_INPUT, 'r', encoding='utf-8') as f: test_input = json.load(f)
TEST_CHECKPOINT_FILE = OUTPUT_DIR / f"{MODEL_KEY}_test_checkpoint.json"

def load_test_checkpoint():
    if TEST_CHECKPOINT_FILE.exists():
        with open(TEST_CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
            data = json.load(f)
            return set(data.get('processed_ids', [])), data.get('predictions', [])
    return set(), []

def save_test_checkpoint(predictions):
    with open(TEST_CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({'processed_ids': [p['id'] for p in predictions], 'predictions': predictions}, f, ensure_ascii=False)

test_processed_ids, test_predictions = load_test_checkpoint()
print(f"Test: {len(test_input)} samples, {len(test_processed_ids)} already done")

for item in tqdm(test_input, desc="Test Set"):
    if item['id'] in test_processed_ids: continue
    try:
        audio_path = TEST_AUDIO_DIR / f"{item['id']}.mp3"
        result = transcribe_diacritized(audio_path, item['text_undiacritized'])
        test_predictions.append({'id': item['id'], 'text_diacritized': result})
        save_test_checkpoint(test_predictions)
    except:
        test_predictions.append({'id': item['id'], 'text_diacritized': item['text_undiacritized']})
        save_test_checkpoint(test_predictions)

# Submission
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
json_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission.json"
with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=2)
zip_file = SUBMISSION_DIR / f"{MODEL_KEY}_submission_{timestamp}.zip"
with zipfile.ZipFile(zip_file, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_file, 'submission.json')
print(f"SUBMISSION: {zip_file}")

Test: 328 samples, 0 already done


Test Set:  13%|█▎        | 44/328 [02:22<17:58,  3.80s/it]

In [ ]:
# Google Drive sync removed for public release
